# Ensemble XGBoost fine-tuné + Prophet fine-tuné (RTE)

Combine les 2 meilleurs modèles issus des notebooks précédents :
- **XGBoost tuné** (`train_models.ipynb`, run MLflow `xgboost_tuned`) — MAPE ≈1.77% sur toute
  l'année 2025 (horizon fixe 24h). Modèle unique, entraîné une fois, déjà loggué via
  `mlflow.xgboost.log_model` — **rechargé** ici via MLflow, pas réentraîné.
- **Prophet "4w_split" tuné** (`train_prophet_sliding_window.ipynb`, section 7.2) — meilleure
  config trouvée : `changepoint_prior_scale=0.35`, `seasonality_prior_scale=1.0`,
  `seasonality_mode="multiplicative"`, MAPE ≈2.52%, évalué sur 2 fenêtres de 2 semaines
  (hiver/été 2025).

**Asymétrie importante** : XGBoost est un objet modèle unique, entraîné une fois — se recharge
trivialement. Prophet en mode "fenêtre glissante" n'a **pas de modèle unique** : un Prophet
différent est réentraîné à chaque cutoff (tous les jours) sur les 4 dernières semaines. Il n'y a
donc rien à "charger" pour Prophet — seule la **config gagnante** (3 hyperparamètres) est
réutilisable, en rejouant la procédure de fit/predict (dupliquée depuis
`train_prophet_sliding_window.ipynb`, comme toute cette étape de reconstruction est autonome dans
ce notebook, cohérent avec la convention déjà suivie dans ce projet).

**Fenêtre de comparaison** : les 2 mêmes fenêtres de 2 semaines (hiver/été 2025) déjà utilisées
pour Prophet — XGBoost (qui a des prédictions sur toute l'année) est restreint à ces mêmes
fenêtres, pour comparer les deux modèles sur un échantillon strictement identique.

**2 méthodes d'ensemble comparées** : moyenne pondérée (`pred = w·xgb + (1-w)·prophet`, `w`
cherché par grid search) et stacking (régression linéaire à 2 features). Avec seulement ~2
semaines de données disponibles par fenêtre, le poids/les coefficients sont appris sur **une**
fenêtre et évalués sur **l'autre** (hiver→été et été→hiver) — comme une validation croisée à 2
plis, pour éviter de sur-apprendre sur un échantillon aussi réduit.

MLflow logge ce notebook dans une expérience dédiée (`rte-consumption-forecast-ensemble`) —
protocole encore différent des 2 précédents, donc pas comparable directement dans la même vue.


## 1. Imports

In [1]:
# Librairies de base
import logging

import numpy as np
import pandas as pd

# Jours fériés français, pour reconstruire les features XGBoost (target_is_holiday)
import holidays

# XGBoost (rechargé, pas réentraîné) ; Prophet (réentraîné à chaque cutoff, config figée)
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

# Coupe le message "Importing plotly failed" (loggé en niveau ERROR par Prophet) dans ce process —
# bruit sans conséquence, voir train_prophet_sliding_window.ipynb pour le détail (réapparaît quand
# même dans les processus worker joblib, accepté comme bruit cosmétique)
logging.getLogger("prophet.plot").setLevel(logging.CRITICAL)
from prophet import Prophet
from joblib import Parallel, delayed
from tqdm import tqdm

# MLflow : rechargement du modèle XGBoost + logging de ce notebook dans sa propre expérience
import mlflow
import mlflow.xgboost

import matplotlib.pyplot as plt


## 2. Configuration

In [2]:
DATA_PATH = "../data/training_dataset_2022_2025.csv"
TARGET_HORIZON_H = 24  # même horizon fixe que train_models.ipynb
LAGS = [1, 24, 48, 168]
TEST_CUTOFF = pd.Timestamp("2025-01-01T00:00:00+00:00")

# Fenêtres de comparaison (mêmes dates que les notebooks précédents). Version tz-aware pour
# XGBoost (target_datetime_utc est en UTC tz-aware dans train_models.ipynb) et version tz-naïve
# pour Prophet (ds est en UTC naïf, voir train_prophet_sliding_window.ipynb pour le pourquoi).
PLOT_WINDOWS = {
    "hiver": (pd.Timestamp("2025-01-06T00:00:00+00:00"), pd.Timestamp("2025-01-20T00:00:00+00:00")),
    "été": (pd.Timestamp("2025-07-07T00:00:00+00:00"), pd.Timestamp("2025-07-21T00:00:00+00:00")),
}
PLOT_WINDOWS_NAIVE = {
    label: (ws.tz_localize(None), we.tz_localize(None)) for label, (ws, we) in PLOT_WINDOWS.items()
}

# Tracking MLflow : même base sqlite locale que tous les notebooks précédents
MLFLOW_TRACKING_URI = "sqlite:///../mlflow.db"
XGB_EXPERIMENT_NAME = "rte-consumption-forecast"                # où se trouve le run xgboost_tuned
ENSEMBLE_EXPERIMENT_NAME = "rte-consumption-forecast-ensemble"  # où ce notebook logge ses résultats

# Régresseurs météo utilisés par Prophet (mêmes 4 colonnes que train_prophet_sliding_window.ipynb)
PROPHET_REGRESSOR_COLUMNS = [
    "temperature_weighted_mean", "temperature_min", "temperature_max", "temperature_std",
]
RETRAIN_INTERVAL = pd.Timedelta(1, unit="D")

# Meilleure config Prophet trouvée en section 7.2 de train_prophet_sliding_window.ipynb
# (trials_4w_split.iloc[0]) — pas de modèle à charger pour Prophet, on rejoue la procédure de
# fit/predict avec ces hyperparamètres
PROPHET_BEST_CONFIG = {
    "train_window": pd.DateOffset(weeks=4),
    "yearly_seasonality": False,
    "weekly_seasonality": False,  # remplacée par la saisonnalité hebdo conditionnelle ci-dessous
    "daily_seasonality": True,
    "weekly_seasonality_split": True,
    "changepoint_prior_scale": 0.35,
    "seasonality_prior_scale": 1.0,
    "seasonality_mode": "multiplicative",
}


## 3. Reconstruction des features XGBoost

Dupliqué de `train_models.ipynb` (sections 3-8, condensé) — même dataset, mêmes lags/features
calendaires/jours fériés, même horizon fixe 24h. Seul `test_df` sera vraiment utilisé (pour les
prédictions XGBoost sur la fenêtre de comparaison), mais reconstruit à l'identique du notebook
source pour éviter tout écart de feature engineering.


In [3]:
hourly = pd.read_csv(DATA_PATH, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()


def lag_by_timestamp(series: pd.Series, hours: int) -> np.ndarray:
    """Valeur de `series` exactement `hours` heures avant chaque timestamp (lookup par timestamp,
    pas par décalage de ligne — voir train_models.ipynb pour le détail)."""
    target_ts = series.index - pd.Timedelta(hours, unit="h")
    return series.reindex(target_ts).to_numpy()


for lag_h in LAGS:
    hourly[f"consumption_lag_{lag_h}"] = lag_by_timestamp(hourly["consumption"], lag_h)
hourly["rolling_mean_24h"] = hourly["consumption"].rolling("24h").mean()
hourly["rolling_std_24h"] = hourly["consumption"].rolling("24h").std()
hourly["rolling_mean_168h"] = hourly["consumption"].rolling("168h").mean()
hourly["temp_t0"] = hourly["temperature_weighted_mean"]

ORIGIN_FEATURE_COLUMNS = [f"consumption_lag_{h}" for h in LAGS] + [
    "rolling_mean_24h", "rolling_std_24h", "rolling_mean_168h", "temp_t0",
]
print(f"{len(hourly)} lignes chargées, de {hourly.index.min()} à {hourly.index.max()}")


35064 lignes chargées, de 2021-12-31 23:00:00+00:00 à 2025-12-31 22:00:00+00:00


In [4]:
supervised_df = hourly[ORIGIN_FEATURE_COLUMNS].copy()
supervised_df.index.name = "origin_datetime_utc"
supervised_df["target_datetime_utc"] = supervised_df.index + pd.Timedelta(TARGET_HORIZON_H, unit="h")
supervised_df["target_consumption"] = hourly["consumption"].reindex(supervised_df["target_datetime_utc"]).to_numpy()
supervised_df["temp_target"] = hourly["temperature_weighted_mean"].reindex(supervised_df["target_datetime_utc"]).to_numpy()
supervised_df = supervised_df.reset_index()

# Features calendaires sur le timestamp CIBLE (heure locale française, jours fériés, encodages
# cycliques) — identique à train_models.ipynb section 6
target_local = supervised_df["target_datetime_utc"].dt.tz_convert("Europe/Paris")
supervised_df["target_local_hour"] = target_local.dt.hour
supervised_df["target_day_of_week"] = target_local.dt.dayofweek
supervised_df["target_month"] = target_local.dt.month
supervised_df["target_is_weekend"] = supervised_df["target_day_of_week"].isin([5, 6]).astype(int)

fr_holidays = holidays.France(years=range(target_local.dt.year.min() - 1, target_local.dt.year.max() + 2))
supervised_df["target_is_holiday"] = target_local.dt.date.isin(fr_holidays).astype(int)

for col, short_name, period in [
    ("target_local_hour", "hour", 24),
    ("target_day_of_week", "dow", 7),
    ("target_month", "month", 12),
]:
    angle = 2 * np.pi * supervised_df[col] / period
    supervised_df[f"target_{short_name}_sin"] = np.sin(angle)
    supervised_df[f"target_{short_name}_cos"] = np.cos(angle)

FEATURE_COLUMNS = ORIGIN_FEATURE_COLUMNS + [
    "temp_target",
    "target_local_hour", "target_day_of_week", "target_month", "target_is_weekend", "target_is_holiday",
    "target_hour_sin", "target_hour_cos",
    "target_dow_sin", "target_dow_cos",
    "target_month_sin", "target_month_cos",
]
model_df = supervised_df.dropna(subset=FEATURE_COLUMNS + ["target_consumption"]).reset_index(drop=True)

train_df = model_df[model_df["target_datetime_utc"] < TEST_CUTOFF].copy()
test_df = model_df[model_df["target_datetime_utc"] >= TEST_CUTOFF].copy()
print(f"Test : {len(test_df)} lignes, cibles de {test_df['target_datetime_utc'].min()} à {test_df['target_datetime_utc'].max()}")


Test : 8759 lignes, cibles de 2025-01-01 00:00:00+00:00 à 2025-12-31 22:00:00+00:00


## 4. Chargement du modèle XGBoost tuné + prédictions sur les fenêtres de comparaison

Recherche dynamique du run `xgboost_tuned` par nom (pas d'ID en dur — robuste à une réexécution
future de `train_models.ipynb`), chargement du modèle natif via `mlflow.xgboost.load_model`, puis
prédiction restreinte aux 2 fenêtres de comparaison (hiver/été 2025).


In [5]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

xgb_exp = mlflow.get_experiment_by_name(XGB_EXPERIMENT_NAME)
xgb_run = mlflow.search_runs(
    experiment_ids=[xgb_exp.experiment_id],
    filter_string="tags.mlflow.runName = 'xgboost_tuned'",
    order_by=["start_time DESC"],
).iloc[0]
xgb_model = mlflow.xgboost.load_model(f"runs:/{xgb_run.run_id}/model")
print(f"Modèle XGBoost tuné chargé depuis le run {xgb_run.run_id}")


def window_label_for(ts, windows):
    for label, (ws, we) in windows.items():
        if ws <= ts < we:
            return label
    return None


xgb_window_df = test_df.copy()
xgb_window_df["window_label"] = xgb_window_df["target_datetime_utc"].apply(
    lambda ts: window_label_for(ts, PLOT_WINDOWS)
)
xgb_window_df = xgb_window_df.dropna(subset=["window_label"]).copy()
xgb_window_df["y_pred_xgb"] = xgb_model.predict(xgb_window_df[FEATURE_COLUMNS])

mape_xgb_raw = mean_absolute_percentage_error(xgb_window_df["target_consumption"], xgb_window_df["y_pred_xgb"]) * 100
print(f"XGBoost tuné — {len(xgb_window_df)} lignes sur les 2 fenêtres, MAPE={mape_xgb_raw:.2f}%")


Modèle XGBoost tuné chargé depuis le run 172c3c3234aa400bb5d66697e80d8ecc
XGBoost tuné — 672 lignes sur les 2 fenêtres, MAPE=1.79%


## 5. Prophet "4w_split" tuné — fit/predict sur les mêmes fenêtres

Dupliqué de `train_prophet_sliding_window.ipynb` (`_fit_and_predict_one_cutoff` et la boucle de
cutoffs, condensés en une seule fonction) — même principe : réentraînement quotidien sur les 4
dernières semaines glissantes, prédiction des 24h suivantes, avec `PROPHET_BEST_CONFIG`.


In [6]:
prophet_df = pd.DataFrame({
    "ds": hourly.index.tz_localize(None),
    "y": hourly["consumption"].to_numpy(),
    "temperature_weighted_mean": hourly["temperature_weighted_mean"].to_numpy(),
    "temperature_min": hourly["temperature_min"].to_numpy(),
    "temperature_max": hourly["temperature_max"].to_numpy(),
    "temperature_std": hourly["temperature_std"].to_numpy(),
})
prophet_df["is_weekday"] = prophet_df["ds"].dt.dayofweek < 5
prophet_df["is_weekend"] = ~prophet_df["is_weekday"]


def _fit_and_predict_one_cutoff(cutoff, config, prophet_df, regressor_columns):
    """Réentraîne Prophet sur les config['train_window'] dernières semaines avant `cutoff`, puis
    prédit les 24h suivantes. Fonction autonome (pas de closure) pour tourner dans un processus
    séparé via joblib."""
    train_slice = prophet_df[
        (prophet_df["ds"] >= cutoff - config["train_window"]) & (prophet_df["ds"] < cutoff)
    ]

    model = Prophet(
        growth="linear",
        daily_seasonality=config["daily_seasonality"],
        weekly_seasonality=config["weekly_seasonality"],
        yearly_seasonality=config["yearly_seasonality"],
        seasonality_mode=config["seasonality_mode"],
        changepoint_prior_scale=config["changepoint_prior_scale"],
        seasonality_prior_scale=config["seasonality_prior_scale"],
    )
    if config.get("weekly_seasonality_split"):
        model.add_seasonality(name="weekly_weekday", period=7, fourier_order=3, condition_name="is_weekday")
        model.add_seasonality(name="weekly_weekend", period=7, fourier_order=3, condition_name="is_weekend")
    model.add_country_holidays(country_name="FR")
    for col in regressor_columns:
        model.add_regressor(col)
    model.fit(train_slice)

    future_ds = pd.date_range(cutoff + pd.Timedelta(1, unit="h"), periods=24, freq="h")
    future = pd.DataFrame({"ds": future_ds}).merge(
        prophet_df[["ds"] + regressor_columns + ["is_weekday", "is_weekend"]], on="ds", how="left"
    )
    forecast = model.predict(future)
    return pd.DataFrame({"ds": future_ds, "yhat": forecast["yhat"].to_numpy(), "cutoff": cutoff})


def run_prophet_window(window_start, window_end, config, desc=None):
    """Cutoffs quotidiens sur [window_start, window_end), parallélisés sur tous les cœurs (comme
    train_prophet_sliding_window.ipynb section 5)."""
    cutoffs = []
    cutoff = window_start
    while cutoff < window_end:
        cutoffs.append(cutoff)
        cutoff += RETRAIN_INTERVAL

    rows = list(tqdm(
        Parallel(n_jobs=-1, return_as="generator_unordered")(
            delayed(_fit_and_predict_one_cutoff)(c, config, prophet_df, PROPHET_REGRESSOR_COLUMNS)
            for c in cutoffs
        ),
        total=len(cutoffs), desc=desc or "Cutoffs",
    ))
    result_df = pd.concat(rows, ignore_index=True).sort_values(["ds", "cutoff"])
    result_df = result_df.drop_duplicates("ds", keep="last")
    result_df = result_df.merge(prophet_df[["ds", "y"]], on="ds", how="left")
    return result_df[(result_df["ds"] >= window_start) & (result_df["ds"] < window_end)]


prophet_results = []
for window_label, (window_start, window_end) in PLOT_WINDOWS_NAIVE.items():
    r = run_prophet_window(
        window_start, window_end, PROPHET_BEST_CONFIG,
        desc=f"Prophet 4w_split_tuned - {window_label}",
    )
    r["window_label"] = window_label
    prophet_results.append(r)
prophet_window_df = pd.concat(prophet_results, ignore_index=True)

mape_prophet_raw = mean_absolute_percentage_error(prophet_window_df["y"], prophet_window_df["yhat"]) * 100
print(f"Prophet 4w_split_tuné — {len(prophet_window_df)} lignes sur les 2 fenêtres, MAPE={mape_prophet_raw:.2f}%")


Prophet 4w_split_tuned - hiver:   0%|          | 0/14 [00:00<?, ?it/s]

Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.
09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdsta

09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdstanpy - INFO - Chain [1] start processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
Prophet 4w_split_tuned - hiver:   7%|▋         | 1/14 [00:00<00:10,  1.22it/s]

09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing


09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
09:49:31 - cmdstanpy - INFO - Chain [1] done processing
Prophet 4w_split_tuned - hiver:  79%|███████▊  | 11/14 [00:01<00:00, 13.91it/s]

...

09:49:32 - cmdstanpy - INFO - Chain [1] done processing
09:49:32 - cmdstanpy - INFO - Chain [1] done processing


09:49:32 - cmdstanpy - INFO - Chain [1] done processing


Prophet 4w_split_tuned - été: 100%|██████████| 14/14 [00:00<00:00, 20.59it/s]

Prophet 4w_split_tuned - été: 100%|██████████| 14/14 [00:00<00:00, 15.96it/s]

Prophet 4w_split_tuné — 670 lignes sur les 2 fenêtres, MAPE=2.52%


## 6. Alignement des prédictions des 2 modèles

Merge des 2 dataframes de résultats sur le timestamp cible commun (XGBoost en UTC tz-aware,
Prophet en UTC naïf — on aligne sur la version naïve). Donne un échantillon strictement identique
pour les 2 modèles, base de tout ce qui suit (MAPE individuels "honnêtes" + ensembling).


In [7]:
xgb_window_df["ds"] = xgb_window_df["target_datetime_utc"].dt.tz_localize(None)

combined_df = xgb_window_df[["ds", "window_label", "target_consumption", "y_pred_xgb"]].merge(
    prophet_window_df[["ds", "yhat"]].rename(columns={"yhat": "y_pred_prophet"}),
    on="ds", how="inner",
)
combined_df = combined_df.rename(columns={"target_consumption": "y_true"})

print(
    f"{len(combined_df)} points alignés "
    f"(sur {len(xgb_window_df)} XGBoost / {len(prophet_window_df)} Prophet côté brut)"
)

mape_xgb = mean_absolute_percentage_error(combined_df["y_true"], combined_df["y_pred_xgb"]) * 100
mape_prophet = mean_absolute_percentage_error(combined_df["y_true"], combined_df["y_pred_prophet"]) * 100
print(f"XGBoost (échantillon aligné)  : MAPE={mape_xgb:.2f}%")
print(f"Prophet (échantillon aligné)  : MAPE={mape_prophet:.2f}%")


670 points alignés (sur 672 XGBoost / 670 Prophet côté brut)
XGBoost (échantillon aligné)  : MAPE=1.78%
Prophet (échantillon aligné)  : MAPE=2.52%


## 7. Ensemble — moyenne pondérée

`pred = w·xgb + (1-w)·prophet`, `w` cherché par grid search sur [0, 1] pour minimiser le MAPE.
Protocole à 2 plis : poids appris sur une fenêtre, évalué sur l'autre (pas de fit et d'évaluation
sur les mêmes données, vu le peu de données disponibles).


In [8]:
def best_weight(df_fit):
    """Cherche le poids w (part de XGBoost) qui minimise le MAPE sur df_fit."""
    weights = np.linspace(0, 1, 101)
    mapes = [
        mean_absolute_percentage_error(
            df_fit["y_true"], w * df_fit["y_pred_xgb"] + (1 - w) * df_fit["y_pred_prophet"]
        ) * 100
        for w in weights
    ]
    best_idx = int(np.argmin(mapes))
    return weights[best_idx], mapes[best_idx]


weighted_avg_results = []
for fit_label, test_label in [("hiver", "été"), ("été", "hiver")]:
    df_fit = combined_df[combined_df["window_label"] == fit_label]
    df_test = combined_df[combined_df["window_label"] == test_label]

    w, mape_fit = best_weight(df_fit)
    pred_test = w * df_test["y_pred_xgb"] + (1 - w) * df_test["y_pred_prophet"]
    mape_test = mean_absolute_percentage_error(df_test["y_true"], pred_test) * 100

    weighted_avg_results.append({
        "fit_on": fit_label, "test_on": test_label, "weight_xgb": w, "mape_test": mape_test,
    })
    print(
        f"Moyenne pondérée — poids appris sur {fit_label} (w_xgb={w:.2f}) "
        f"→ MAPE sur {test_label} = {mape_test:.2f}%"
    )

weighted_avg_df = pd.DataFrame(weighted_avg_results)
weighted_avg_df


Moyenne pondérée — poids appris sur hiver (w_xgb=0.82) → MAPE sur été = 1.59%
Moyenne pondérée — poids appris sur été (w_xgb=0.61) → MAPE sur hiver = 1.72%


,fit_on,test_on,weight_xgb,mape_test
0,hiver,été,0.82,1.586596
1,été,hiver,0.61,1.719479


## 8. Ensemble — stacking (régression linéaire)

`y ≈ a·xgb_pred + b·prophet_pred + c`, coefficients appris par `LinearRegression`. Même protocole
à 2 plis que la moyenne pondérée, pour comparer directement les deux méthodes sur un pied
d'égalité.


In [9]:
stacking_results = []
for fit_label, test_label in [("hiver", "été"), ("été", "hiver")]:
    df_fit = combined_df[combined_df["window_label"] == fit_label]
    df_test = combined_df[combined_df["window_label"] == test_label]

    meta_model = LinearRegression()
    meta_model.fit(df_fit[["y_pred_xgb", "y_pred_prophet"]], df_fit["y_true"])
    pred_test = meta_model.predict(df_test[["y_pred_xgb", "y_pred_prophet"]])
    mape_test = mean_absolute_percentage_error(df_test["y_true"], pred_test) * 100

    stacking_results.append({
        "fit_on": fit_label, "test_on": test_label,
        "coef_xgb": meta_model.coef_[0], "coef_prophet": meta_model.coef_[1],
        "intercept": meta_model.intercept_, "mape_test": mape_test,
    })
    print(
        f"Stacking — appris sur {fit_label} (coef_xgb={meta_model.coef_[0]:.2f}, "
        f"coef_prophet={meta_model.coef_[1]:.2f}, intercept={meta_model.intercept_:.0f}) "
        f"→ MAPE sur {test_label} = {mape_test:.2f}%"
    )

stacking_df = pd.DataFrame(stacking_results)
stacking_df


Stacking — appris sur hiver (coef_xgb=0.77, coef_prophet=0.27, intercept=-2683) → MAPE sur été = 2.43%
Stacking — appris sur été (coef_xgb=0.72, coef_prophet=0.30, intercept=-1197) → MAPE sur hiver = 1.81%


,fit_on,test_on,coef_xgb,coef_prophet,intercept,mape_test
0,hiver,été,0.773776,0.265245,-2682.627319,2.434140
1,été,hiver,0.722859,0.303742,-1196.520355,1.806862


## 9. Tableau comparatif final + logging MLflow

Modèles individuels vs les 2 méthodes d'ensemble (dans les 2 sens hiver/été), loggués dans un run
`ensemble_summary` de l'expérience dédiée `rte-consumption-forecast-ensemble` (tableau complet +
détail des poids/coefficients en artifacts CSV).


In [10]:
comparison_rows = [
    {"method": "xgboost_alone", "mape": mape_xgb},
    {"method": "prophet_alone", "mape": mape_prophet},
    {"method": "weighted_avg_fit_hiver_test_ete", "mape": weighted_avg_df.iloc[0]["mape_test"]},
    {"method": "weighted_avg_fit_ete_test_hiver", "mape": weighted_avg_df.iloc[1]["mape_test"]},
    {"method": "stacking_fit_hiver_test_ete", "mape": stacking_df.iloc[0]["mape_test"]},
    {"method": "stacking_fit_ete_test_hiver", "mape": stacking_df.iloc[1]["mape_test"]},
]
comparison_df = pd.DataFrame(comparison_rows).sort_values("mape").reset_index(drop=True)

mlflow.set_experiment(ENSEMBLE_EXPERIMENT_NAME)
with mlflow.start_run(run_name="ensemble_summary"):
    mlflow.set_tags({"dataset_version": "2022_2025", "approach": "ensemble_xgboost_prophet"})
    mlflow.log_params({
        "xgb_run_id": xgb_run.run_id,
        "prophet_config": str(PROPHET_BEST_CONFIG),
    })
    for _, row in comparison_df.iterrows():
        mlflow.log_metric(f"mape_{row['method']}", row["mape"])
    mlflow.log_text(comparison_df.to_csv(index=False), "comparison_results.csv")
    mlflow.log_text(weighted_avg_df.to_csv(index=False), "weighted_avg_results.csv")
    mlflow.log_text(stacking_df.to_csv(index=False), "stacking_results.csv")

comparison_df


2026/08/03 09:49:32 INFO mlflow.tracking.fluent: Experiment with name 'rte-consumption-forecast-ensemble' does not exist. Creating a new experiment.


,method,mape
0,weighted_avg_fit_hiver_test_ete,1.586596
1,weighted_avg_fit_ete_test_hiver,1.719479
2,xgboost_alone,1.782644
3,stacking_fit_ete_test_hiver,1.806862
4,stacking_fit_hiver_test_ete,2.434140
5,prophet_alone,2.520662


## 10. Conclusion

- **Comparaison sur échantillon strictement identique** (2×2 semaines, hiver/été 2025) :
  XGBoost seul vs Prophet seul vs 2 méthodes d'ensemble (moyenne pondérée, stacking), chacune
  évaluée dans les 2 sens (fit hiver/test été et inversement) pour éviter le sur-apprentissage sur
  un si petit échantillon.
- **Question centrale** : l'ensemble apporte-t-il un vrai gain par rapport à XGBoost seul (déjà
  nettement meilleur que Prophet), ou le poids optimal converge-t-il simplement vers ~1 (XGBoost
  domine, Prophet n'apporte pas de signal complémentaire) ? Voir le tableau comparatif ci-dessus.
- **Limitation principale** : poids/coefficients estimés sur un échantillon très réduit (~2
  semaines par pli) — un résultat prometteur mériterait d'être confirmé sur une fenêtre de
  comparaison plus large (nécessiterait d'étendre l'évaluation Prophet au-delà des 2×2 semaines
  actuelles, plus coûteux en calcul).
- **Limitations héritées des notebooks sources** : `temp_target`/régresseurs météo Prophet
  utilisent la météo **réelle**, pas des prévisions archivées (même limitation que les 3 notebooks
  précédents).
